# Project: QA Bug Triage Agent
**Brief from Paras:**
Watches Gmail/Slack for bug reports, classifies severity,
checks Linear/Jira for duplicates, attaches screenshots from Drive, creates ticket.

This is a SCAFFOLD. The supervisor + graph wiring is complete; worker logic for some nodes is marked TODO.
Use Project #2 (`02_support_resolver/support_resolver.ipynb`) as your reference implementation.

## Submission checklist
- [ ] All worker TODOs filled in
- [ ] Composio actions verified against docs.composio.dev
- [ ] HITL where destructive actions occur
- [ ] Checkpoint persistence configured
- [ ] Graph diagram saved as graph.png
- [ ] README.md with architecture + example run


## 1. Setup

In [ ]:
import os, sqlite3
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(".env")

assert os.getenv("OPENAI_API_KEY")
assert os.getenv("COMPOSIO_API_KEY"), "Sign up at composio.dev and connect required apps in their dashboard"
print("env OK")


## 2. State schema

In [ ]:
from typing import TypedDict, Annotated, Optional, Literal
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage, SystemMessage

class QaBugTriageState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    next_worker: str
    bug_report: str
    severity: Optional[str]
    duplicate_id: Optional[str]
    screenshot_link: Optional[str]
    linear_ticket_id: Optional[str]


## 3. LLM and Composio toolset

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from composio_langgraph import Action, ComposioToolSet

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
toolset = ComposioToolSet()


## 4. Workers

In [ ]:
# classifier - Severity classifier (pure LLM)
# No Composio actions; pure LLM or custom logic.
# TODO: implement classifier_node(state) following the pattern in 02_support_resolver.
def classifier_node(state):
    """Classify bug as P0 (critical), P1 (high), P2 (medium), P3 (low) based on impact and urgency."""
    raise NotImplementedError('TODO: implement classifier_node')


# dedup_checker - Linear duplicate finder
dedup_checker_tools = toolset.get_tools(actions=[Action.LINEAR_SEARCH_ISSUES])
dedup_checker_agent = create_react_agent(llm, dedup_checker_tools, prompt='''Search Linear for existing issues with similar title/body. Return duplicate_id if found.''')
def dedup_checker_node(state):
    result = dedup_checker_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='dedup_checker')]}


# screenshot_fetcher - Drive screenshot finder
screenshot_fetcher_tools = toolset.get_tools(actions=[Action.GOOGLEDRIVE_FIND_FILE])
screenshot_fetcher_agent = create_react_agent(llm, screenshot_fetcher_tools, prompt='''Find latest screenshot in 'Bug Reports' folder by reporter name.''')
def screenshot_fetcher_node(state):
    result = screenshot_fetcher_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='screenshot_fetcher')]}


# ticket_creator - Linear ticket creator
ticket_creator_tools = toolset.get_tools(actions=[Action.LINEAR_CREATE_LINEAR_ISSUE])
ticket_creator_agent = create_react_agent(llm, ticket_creator_tools, prompt='''Create Linear issue with severity label, screenshot link, and assignee per Notion routing rules.''')
def ticket_creator_node(state):
    result = ticket_creator_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='ticket_creator')]}


# notifier - Reporter notifier
notifier_tools = toolset.get_tools(actions=[Action.SLACK_SENDS_A_MESSAGE_TO_A_SLACK_CHANNEL])
notifier_agent = create_react_agent(llm, notifier_tools, prompt='''DM the reporter with ticket link or duplicate reference.''')
def notifier_node(state):
    result = notifier_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='notifier')]}


## 5. Supervisor + router

In [ ]:
def supervisor(state) -> dict:
    if state.get('severity') is None: return {'next_worker': 'classifier'}
    if state.get('duplicate_id') is None and 'dedup_checked' not in str(state['messages']):
        return {'next_worker': 'dedup_checker'}
    if state['duplicate_id']: return {'next_worker': 'notifier'}
    if state.get('screenshot_link') is None: return {'next_worker': 'screenshot_fetcher'}
    if state.get('linear_ticket_id') is None: return {'next_worker': 'ticket_creator'}
    if 'reporter notified' not in (state['messages'][-1].content.lower() if state['messages'] else ''):
        return {'next_worker': 'notifier'}
    return {'next_worker': 'DONE'}

def route(state) -> str:
    nxt = state['next_worker']
    if nxt in {'ticket_creator', 'classifier', 'dedup_checker', 'screenshot_fetcher', 'notifier'}:
        return nxt
    return '__end__'


## 6. Build graph + checkpoint persistence

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver

g = StateGraph(globals()[[k for k in globals() if k.endswith('State') and k != 'AgentState'][0]])
g.add_node("supervisor", supervisor)
g.add_node("classifier", classifier_node)
g.add_node("dedup_checker", dedup_checker_node)
g.add_node("screenshot_fetcher", screenshot_fetcher_node)
g.add_node("ticket_creator", ticket_creator_node)
g.add_node("notifier", notifier_node)

g.add_edge(START, "supervisor")
g.add_conditional_edges("supervisor", route, {
    "classifier": "classifier",
    "dedup_checker": "dedup_checker",
    "screenshot_fetcher": "screenshot_fetcher",
    "ticket_creator": "ticket_creator",
    "notifier": "notifier",
    "__end__": END,
})
g.add_edge("classifier", "supervisor")
g.add_edge("dedup_checker", "supervisor")
g.add_edge("screenshot_fetcher", "supervisor")
g.add_edge("ticket_creator", "supervisor")
g.add_edge("notifier", "supervisor")

conn = sqlite3.connect("07_qa_bug_triage.db", check_same_thread=False)
app = g.compile(checkpointer=SqliteSaver(conn))


## 7. Visualise (submission requirement)

In [ ]:
from IPython.display import Image, display
try:
    png = app.get_graph().draw_mermaid_png()
    Path("graph.png").write_bytes(png)
    display(Image("graph.png"))
except Exception as e:
    print("ASCII fallback:")
    print(app.get_graph().draw_ascii())


## 8. End-to-end demo

In [ ]:
config = {'configurable': {'thread_id': '07_qa_bug_triage-demo-1'}, 'recursion_limit': 30}

result = app.invoke(
    {'bug_report': 'Login button does not respond on Safari mobile, iOS 17',
        'messages': [HumanMessage(content='New bug report from QA channel')]},
    config=config,
)

print("=== FINAL STATE ===")
for k, v in result.items():
    if k != 'messages':
        print(f"{k}: {str(v)[:150]}")
print("\n=== MESSAGE TRACE ===")
for m in result['messages']:
    label = getattr(m, 'name', m.type)
    print(f"[{label}] {m.content[:200]}")
